# Specialization and detection under stimulation

Six experiments, in dependency order. Roughly 4 hours of MATLAB in total, all cached, so
you can stop after any section and pick up later.

| § | Experiment | MATLAB |
|---|---|---|
| 2 | Does detection survive stimulation at all? | ~37 min |
| 3 | Train a T3P agent per severity (0.3, 0.6, 1.0) | ~24 min |
| 4 | Sweep every agent across seven brain severities | ~68 min |
| 5 | Detection with a real agent running | ~28 min |
| 6 | Smaller step: 0.5 → 0.6 | ~28 min |
| 7 | Cost of letting the agent re-learn instead | ~60 min |

Section 2 needs no trained agent and tests the assumption the whole paper rests on, so it
goes first. If separation collapses there, sections 3 and 4 still stand but the claim
changes, and better to know on day one.

Run from the workspace root. `experiments.py`, `pd_switch.py`, `dbs_t3p.py` and
`helpers.py` all need to be importable, and MATLAB needs `matlab/` and `gating/`.

**Run this notebook from the repo root.** Modules live in `src/`, so the cell below puts that on the import path. The working directory must stay the repo root regardless — MATLAB writes `bgn_vars.mat` there and resolves `matlab/` and `gating/` against it.

Simulation caches resolve to `cache/` automatically; code refers to them by bare name and `src/paths.py` maps them.

In [ ]:
import sys
sys.path.insert(0, "src")

## 1. Setup

One MATLAB engine for everything. `bgn_vars.mat` is a single shared state file, so a
second engine would corrupt the first. `dbs_t3p.MatlabBackend` owns it and the
`pd_switch` functions borrow `backend.eng`.

In [ ]:
%matplotlib inline

import sys, os
from pathlib import Path
print(sys.executable)
print(os.getcwd())
import matlab.engine
print("matlab.engine ok")

Imports. `importlib.reload` is here because editing a `.py` file mid-session otherwise leaves objects bound to stale classes — the usual symptom is an `AttributeError` on a method you can see in the file.

In [ ]:
import importlib
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import helpers
import dbs_t3p as t3p
import pd_switch as ps
import experiments as ex
for m in (t3p, ps, ex):
    importlib.reload(m)

backend = t3p.MatlabBackend(code_dir=".", parent_dir="matlab")
backend.eng.addpath(str(Path("gating").resolve()), nargout=0)
eng = backend.eng
print("engine up, gating on path")

Confirm the gating functions are on the MATLAB path. If this errors, `bgn_step` will fail later in a much less obvious way.

In [ ]:
print(eng.which("stn_minf"))

`MatlabBackend` looks for gating at `matlab/gating`, which does not exist here — yours is
at the repo root. The explicit `addpath` above fixes the warning it prints and makes sure
the gating functions are actually reachable.

## 2. Does detection survive stimulation?

Every detection result so far used `amp = 0`. The paper claims we can spot a severity
change *while a controller is running*, so this is the assumption to test first.

No trained agent needed: a T3P agent's output is a fixed frequency and amplitude, so
"stimulation is running" just means a non-zero `amp` passed to the usual switch run.

10 seeds at each of four amplitudes, about 37 minutes.

In [ ]:
stim_sweep = {}
for amp in (0.0, 250.0, 1000.0, 3500.0):
    res, _ = ps.run_seeds(eng, range(1, 11), helpers.calculate_pb,
                          out_dir=f"runs_stim_a{int(amp)}", freq=130.0, amp=amp)
    stim_sweep[amp] = res
    ex.report_detection(f"amp = {amp:.0f} uA", res)

The raw traces behind that table, one panel per amplitude, each normalised by its own pre-switch mean so the shapes are comparable. The switch is the red line.

**Look at the high-amplitude panels.** Consecutive points alternate high–low. That alternation is what collapses the separation statistic in the table above, and the next cell is the fix.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 10), sharex=True)
for ax, amp in zip(axes, sorted(stim_sweep)):
    t, v = stim_sweep[amp][1]
    ax.plot(t, v / v[:17].mean(), lw=1.4, color="tab:purple")
    ax.axvline(5.0, ls="--", color="tab:red")
    ax.set_ylabel(f"{amp:.0f} uA")
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("time (s)")
fig

What to read. `gap` is how much P_β rises across the switch, `separation` is that rise in
units of within-state noise, `worst` is the hardest of the ten brains, and `detected` is
how many the changepoint test caught at the current threshold.

If separation holds near its unstimulated value out to realistic amplitudes, the full-loop
claim is safe. If it collapses at 250 µA, the paper becomes "specialized controllers work,
and here is the measured limit on detecting a switch while they run" — still a result,
just a different one, and worth knowing now rather than in three weeks.

130 Hz is used here as a stand-in. Section 5 repeats this with the real agent's setting.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
amps = sorted(stim_sweep)
means = [ps.summarize(stim_sweep[a])[1].mean() for a in amps]
worst = [abs(ps.summarize(stim_sweep[a])[1]).min() for a in amps]
ax.plot(amps, means, marker="o", lw=2, label="mean over 10 brains")
ax.plot(amps, worst, marker="s", lw=1.5, ls="--", label="worst brain")
ax.axhline(3, color="tab:green", ls=":", lw=1.2)
ax.text(amps[0], 3, " clustering reliable above here", va="bottom", fontsize=9,
        color="tab:green")
ax.set_xlabel("stimulation amplitude (uA)")
ax.set_ylabel("separation (sd)")
ax.set_title("Does the severity change stay visible under stimulation?")
ax.legend()
ax.grid(alpha=0.3)
fig

**Pair-averaging.** Beta power is computed on a 1000 ms window stepped 250 ms. At high amplitude consecutive windows alternate, and because separation divides the change by the within-segment spread, a regular alternation inflates the denominator while leaving both segment means untouched — so the signal vanishes under a yardstick that grew.

Averaging each window with its neighbour cancels an alternation exactly, while genuine noise only falls by √2. The `gain` figures below are the diagnostic: about 1.2× means no artifact, 2× or more means there is one.

In [ ]:
import numpy as np

def pair_average(res):
    out = {}
    for s, (t, v) in res.items():
        m = (len(v) // 2) * 2
        out[s] = (t[:m].reshape(-1, 2).mean(1), v[:m].reshape(-1, 2).mean(1))
    return out

def separation(res, switch_s=5.0, margin=0.75):
    gaps, seps = [], []
    for t, v in res.values():
        b, a = v[t <= switch_s - margin], v[t >= switch_s + margin]
        gaps.append(100 * (a.mean() / b.mean() - 1))
        seps.append((a.mean() - b.mean()) / np.sqrt((b.var() + a.var()) / 2))
    return np.array(gaps), np.array(seps)

for amp in sorted(stim_sweep):
    for label, r in (("raw", stim_sweep[amp]), ("paired", pair_average(stim_sweep[amp]))):
        g, s = separation(r)
        print(f"amp {amp:6.0f}  {label:<7} gap {g.mean():+6.1f}%  "
              f"separation {s.mean():5.2f} sd  worst {s.min():5.2f}")
    print()

## 3. Train an agent per severity

Three severities: 0.3, 0.6 and 1.0 — the same values the existing reward-model datasets
were collected at. That makes a free cross-check available later: does the bandit converge
on the setting the reward model picks at each severity? Two independent methods agreeing is
worth reporting, and disagreeing is worth more.

T3P over a 49-arm grid: 6 frequencies by 8 non-zero amplitudes, plus a no-stimulation arm.

The default current grid in `dbs_t3p` starts at 1000 µA, which leaves everything below
that unsampled — and the reward-model work found reward falls monotonically with
amplitude, so the interesting region is exactly the one the default grid skips. The fine
grid below is the one from `dbs_rerun.ipynb` cell 42.

In [ ]:
ex.use_fine_grid()

Train one T3P agent per severity. `report_agents` checks two failure modes automatically: all severities choosing the same arm (which would mean a selector has nothing to select between) and any selection sitting on a grid edge.

In [ ]:
agents = ex.train_agents(backend, helpers, pd_values=(0.3, 0.6, 1.0), tmax=1000)
print()
flagged = ex.report_agents(agents)

**Grid-edge check.** The frequency grid is extended to 255 Hz and the current grid made finer, then swept at the mildest and most severe brains. If the optimum moves outward with the grid, the previous optimum was the boundary rather than a result — which has happened repeatedly in this project.

In [ ]:
t3p.FREQ_VALUES = [55, 80, 105, 130, 155, 180, 205, 230, 255]
ex.use_fine_grid()

sweeps = {}
for pd in (0.3, 1.0):
    cfg = t3p.T3PConfig(pd_value=pd, tmax=1000, n_arms=len(t3p.ARMS))
    sweeps[pd] = t3p.sweep_all_arms(backend, cfg, helpers, repeats=3)

Detection at the setting the agents actually converged on, 180 Hz / 500 µA, raw against pair-averaged.

In [ ]:
res500, _ = ps.run_seeds(eng, range(1, 11), helpers.calculate_pb,
                         out_dir="runs_stim_a500", freq=180.0, amp=500.0)
for label, r in (("raw", res500), ("paired", pair_average(res500))):
    g, s = separation(r)
    print(f"500 uA  {label:<7} gap {g.mean():+6.1f}%  separation {s.mean():5.2f} sd  worst {s.min():5.2f}")

What the switch run is actually measuring: the mean level either side of the switch, excluding the window that straddles it.

In [ ]:
t, v = stim_sweep[500.0][1] if 500.0 in stim_sweep else res500[1]
print(f"switch run, before (pd=0.3): pb {v[t <= 4.5].mean():8.0f}")
print(f"switch run, after  (pd=0.7): pb {v[t >= 5.5].mean():8.0f}")

The same severities measured from a **fresh initialisation** rather than across a switch. If the post-switch level sits above the fresh steady state, part of what the detector sees is a transient rather than the new resting level.

In [ ]:
from dataclasses import replace

cfg = t3p.T3PConfig(tmax=1000, n_arms=len(t3p.ARMS))
for pd in (0.3, 0.7, 1.0):
    c = replace(cfg, pd_value=pd)
    pbs = [t3p.compute_reward(*backend.simulate(180, 500, c), 180, 500, c, helpers)["pb"]
           for _ in range(8)]
    print(f"fresh init, pd={pd}: pb {np.mean(pbs):8.0f} +/- {np.std(pbs)/np.sqrt(8):6.0f}")

Beta power against severity at the agents' operating point, eight repeats each.

This is the measurement that decides whether a selector can read severity off the biomarker. Watch for non-monotonicity — if two different severities give the same value, the mapping cannot be inverted.

In [ ]:
for pd in (0.3, 0.4, 0.5, 0.6, 0.7, 0.85, 1.0):
    c = replace(cfg, pd_value=pd)
    pbs = [t3p.compute_reward(*backend.simulate(180, 500, c), 180, 500, c, helpers)["pb"]
           for _ in range(8)]
    print(f"pd={pd:4.2f}  pb {np.mean(pbs):8.0f} +/- {np.std(pbs)/np.sqrt(8):6.0f}")

**Is the measurement window inside the settling transient?** T3P reads its reward over the first second after `bgn_init`. This compares that against a window taken after the model has settled. A large difference means every arm sweep and every trained agent was scored on settling-phase data.

In [ ]:
import scipy.io
c = replace(cfg, pd_value=0.7, tmax=1000)
short = [t3p.compute_reward(*backend.simulate(180, 500, c), 180, 500, c, helpers)["pb"]
         for _ in range(8)]

settled = []
for _ in range(8):
    backend.eng.bgn_init(0.7, 3000.0, int(np.random.randint(9_999_999)), nargout=0)
    backend.eng.bgn_step(180.0, 500.0, 300_000.0, nargout=2)
    vgi = scipy.io.loadmat("bgn_vars.mat")["vgi"]
    settled.append(float(helpers.calculate_pb(vgi[:, 200_000:300_000])))

print(f"measured 0-1 s (as T3P does): {np.mean(short):8.0f} +/- {np.std(short)/np.sqrt(8):6.0f}")
print(f"measured 2-3 s (settled):     {np.mean(settled):8.0f} +/- {np.std(settled)/np.sqrt(8):6.0f}")

Re-score every cached stimulated run with pair-averaging, against a threshold calibrated on the unstimulated brains and then frozen. Reads the caches only — no simulation.

In [ ]:
import recheck_stim
recheck_stim.run()

Latency, raw against paired. Pair-averaging halves the time resolution, so this is where that cost shows up.

In [ ]:
import paths
import numpy as np
from pathlib import Path
import pd_switch as pdsw
import recheck_stim as rs

cal = {s: v for s, v in rs.load_cached("runs_switch").items() if s <= 10}
TH = rs.calibrate(cal, verbose=False)

def fmt(r):
    if not r["ok"]:
        return "-"
    if r["ok"] == 1:
        return f"{r['latency']:.2f}s"
    return f"{r['latency']:.2f}s +/- {r['sem']:.2f}"

print(f"paired threshold {TH}\n")
print(f"{'run':<34}{'raw':>8}   {'latency':<18}{'pair':>6}   {'latency':<18}")
print("-" * 92)
for d in paths.cache_dirs("runs_stim_*"):
    raw = rs.load_cached(d)
    if not raw:
        continue
    par = rs.pair_average(raw)
    rr = pdsw.evaluate(raw, pdsw.detect_cp, thresh=7.0)
    rp = pdsw.evaluate(par, pdsw.detect_cp, thresh=TH, min_size=3)
    print(f"{d.name:<34}{rr['ok']:>3}/{rr['n']:<4}   {fmt(rr):<18}"
          f"{rp['ok']:>2}/{rp['n']:<3}   {fmt(rp):<18}")

A longer record at the worst amplitude. With only 5 s of post-switch data a detector that needs 3 s is pressed against the end of the recording, so its miss rate depends partly on how long we happened to record. Ten seconds removes that constraint.

In [ ]:
res_long, _ = ps.run_seeds(eng, range(1, 11), helpers.calculate_pb,
                           out_dir="runs_stim_long_a3500",
                           freq=130.0, amp=3500.0, hold_ms=10000)

par_long = rs.pair_average(res_long)
sep = ps.summarize(par_long, switch_s=10.0)[1]
r = ps.evaluate(par_long, ps.detect_cp, thresh=TH, min_size=3, switch_s=10.0)
print(f"separation {sep.mean():.2f} sd")
ps.report("3500 uA, 10 s post-switch", r)

**Does the detector need a long baseline?** Truncates the observation history before the switch and re-scores. If latency is flat across baseline lengths, the detector needs no warm-up to reach full performance.

In [ ]:
def truncate_baseline(res, keep_s, switch_s):
    out = {}
    for s, (t, v) in res.items():
        m = t >= (switch_s - keep_s)
        out[s] = (t[m], v[m])
    return out

for keep in (10.0, 7.5, 5.0, 3.0):
    tr = truncate_baseline(res_long, keep, switch_s=10.0)
    par = rs.pair_average(tr)
    r = ps.evaluate(par, ps.detect_cp, thresh=TH, min_size=3, switch_s=10.0)
    lat = f"{r['latency']:.2f}s +/- {r['sem']:.2f}" if r["ok"] > 1 else "-"
    print(f"baseline {keep:4.1f}s   ok {r['ok']:2d}/{r['n']}  "
          f"false {len(r['false'])}  missed {len(r['missed'])}  latency {lat}")

Two failure modes the report checks for automatically.

**All three severities choosing the same arm** would mean a selector has nothing to select
between, and the project's premise is wrong. That is the single most important line of
output in this notebook.

**A grid-edge selection** — top frequency, top amplitude, or no stimulation at all. This
project has produced an apparent optimum sitting on a grid boundary four times now, and an
optimum at the edge usually means the real one is outside the box. If something is flagged,
widen that dimension and retrain before using the agent for anything downstream.

## 4. Cross-controller matrix

Every agent, swept across seven brain severities — including four nobody trained on.
5 seconds per run, 10 seeds, the same seeds for the same brain across agents so the
comparisons are paired.

Testing only at the agents' own severities would tell you the diagonal wins. Sweeping tells
you **how far from its training severity a controller stays competent**, and where two
curves cross is where a selector should switch between them. That is the figure.

It also answers the question a reviewer will ask: how many controllers does the library
actually need? Flat, overlapping curves mean three is too many. A sharp fall two steps from
the training point means three is too few.

About 68 minutes.

In [ ]:
BRAINS = (0.3, 0.4, 0.5, 0.6, 0.7, 0.85, 1.0)

rows = ex.cross_matrix(backend, helpers, agents, brain_pds=BRAINS,
                       seeds=range(1, 11), tmax=5000)
print()
ex.report_matrix(rows)

Degradation curves. Each agent's reward across brains of every severity.

In [ ]:
fig = ex.plot_matrix(rows)
fig

Each curve should peak near its own training severity, marked by the dotted line in its
colour, and fall away on either side. The paired table is the evidence behind the picture:
for every brain it compares the nearest-trained agent against the others on identical
seeds and says whether the difference clears two standard errors.

Brains at 0.4, 0.5, 0.7 and 0.85 have no agent of their own, so they are matched to the
nearest one — those rows are the interpolation test.

"not separated" everywhere is a negative result and a publishable one — it would say the
controllers are interchangeable at these severities, which kills the selector but is worth
reporting honestly.

The same matrix on the raw biomarker rather than reward, which separates how much of any difference is biology and how much is the energy terms.

In [ ]:
ex.report_matrix(rows, key="pb")

The same matrix on the raw biomarker rather than reward. Worth looking at because reward
mixes P_β with an energy penalty, and the earlier reward models collapsed onto
no-stimulation precisely because the energy term dominated. If the diagonal is strong in
P_β but weak in reward, the reward weighting is the problem rather than the controllers.

## 5. Detection with a real agent running

Section 2 used 130 Hz as a placeholder. This repeats it with the pd = 0.3 agent's actual
setting, which is the deployment case: the patient is on their 0.3 controller, their state
changes to 0.7, and nobody has told the controller yet.

30 seeds, about 28 minutes.

In [ ]:
res_stim = ex.detect_under_stim(eng, helpers, agents[0.3],
                                seeds=range(1, 31), pd_start=0.3, pd_end=0.7)
ex.report_detection("0.3 -> 0.7, agent 0.3 running", res_stim)

fig = ps.plot_seeds(res_stim, out_png="pd_switch_under_stim.png")
fig

Compare this against the unstimulated 30-seed result from `pd_switch.ipynb`: +23.4% gap,
5.10 sd separation, 19/20 held out. The difference between the two is the cost of running
a controller, and it is a figure in the paper either way.

## 6. A smaller step: 0.5 → 0.6

Expect this to fail, and plan to report it as the measured limit of the method.

The worst brain at 0.3 → 0.7 was already 1.67 sd, where any detector deciding from one
window at a time mislabels about one window in five. A 0.5 → 0.6 step is a fraction of
that size. A stated boundary is a result; a boundary a reviewer finds is a problem.

15 seeds rather than 30, because establishing a failure does not need the same precision as
establishing a result. If it unexpectedly works, re-run at 30 before reporting it.

The 0.6 agent runs here rather than the 0.3 one, since 0.6 is the nearest trained
controller to a patient sitting between 0.5 and 0.6.

In [ ]:
res_small, _ = ps.run_seeds(eng, range(1, 16), helpers.calculate_pb,
                            out_dir="runs_switch_05to06",
                            pd_start=0.5, pd_end=0.6, freq=130.0, amp=0.0)
ex.report_detection("0.5 -> 0.6, no stimulation", res_small)

The same small step with the matching agent running.

In [ ]:
res_small_stim = ex.detect_under_stim(eng, helpers, agents[0.6],
                                      seeds=range(1, 16), pd_start=0.5, pd_end=0.6)
ex.report_detection("0.5 -> 0.6, agent 0.6 running", res_small_stim)

## 7. What it costs to let the agent re-learn

The alternative to detect-and-switch: keep the bandit live, and when the brain changes let
its own falling rewards push it to new parameters. Two things to measure — how fast it
notices, and what the noticing costs the patient.

The published `T3PBandit` cannot do this, for three reasons, so `AdaptiveT3P` overrides it:

- **`Q += (r - Q) / counts`** averages over every pull ever, so after 60 pulls a new reward
  moves Q by under 2%. Replaced with a constant step size, `Q += alpha * (r - Q)`.
- **Epsilon decays to zero** by round 8, after which no other arm is ever sampled and their
  Q values stay frozen at their pre-change estimates. Replaced with a floor.
- **Pruning is permanent.** The top-25 is chosen on pre-change performance, so the arm that
  is best afterwards may already have been discarded. Disabled.

`dbs_t3p.py` is untouched.

The most expensive section, roughly 20 minutes per seed. Start with 3 and add more only if
the spread is wide.

In [ ]:
TARGET = ex.arm_index(agents[1.0]["freq"], agents[1.0]["curr"])
ORACLE = np.mean([r["reward"] for r in rows
                  if r["agent_pd"] == 1.0 and r["brain_pd"] == 1.0])
print(f"target arm {TARGET} = {t3p.ARMS[TARGET]}, oracle reward {ORACLE:+.4f}")

adaptive = [ex.run_adaptive_switch(backend, helpers, pd_before=0.3, pd_after=1.0,
                                   rounds_before=50, rounds_after=200, tmax=1000,
                                   target_arm=TARGET, oracle_reward=ORACLE,
                                   seed=s, eps_floor=0.30)
            for s in (1, 2, 3)]
print()
ex.report_adaptive(adaptive)

The adaptive run. The agent is left on a brain whose severity changed and has to notice from its own falling rewards.

In [ ]:
fig = ex.plot_adaptive(adaptive, oracle_reward=ORACLE)
fig

The two vertical markers are the whole result. The agent drops its old setting almost
immediately — that is a usable change signal, and it is what you meant by measuring a
change in stimulation. Finding *good* new parameters takes far longer, and the gap between
the two lines is stimulation the patient receives while the agent is still searching.

On a toy landscape the agent took around 115 pulls to re-optimise. If the real one is
similar, that is roughly two minutes of brain time on wrong settings — which is the number
that justifies a pre-trained library, and the baseline the detect-and-switch result should
be compared against.

Caveats worth stating in the paper: `eps_floor`, `alpha` and the number of post-change
rounds all affect how fast it recovers, and with too few rounds it never recovers at all.
Sweep them rather than reporting one setting.

## 8. Summary

**Summary.** Everything above in one block.

In [ ]:
print("agents")
ex.report_agents(agents)

print("\n\ndetection")
ex.report_detection("0.3 -> 0.7, no stim", stim_sweep[0.0])
ex.report_detection("0.3 -> 0.7, agent running", res_stim)
ex.report_detection("0.5 -> 0.6, no stim", res_small)
ex.report_detection("0.5 -> 0.6, agent running", res_small_stim)

print("\n\nspecialization")
ex.report_matrix(rows)

print("\n\nletting the agent re-learn instead")
ex.report_adaptive(adaptive)

Shut the engine down. Do this before closing the notebook, or the MATLAB process is left running.

In [ ]:
backend.close()
print("engine closed")